# Компьютерное моделирование влияния осадков на возникновение паводков в районах Кыргызстана

## Демонстрационный ноутбук

Данный ноутбук демонстрирует основные возможности разработанной системы моделирования паводков.

## 1. Импорт библиотек и настройка

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Импорт модулей проекта
from src.config import REGIONS, FloodThresholds
from src.data.precipitation import PrecipitationDataLoader, generate_synthetic_precipitation
from src.data.terrain import TerrainAnalyzer, create_sample_terrain
from src.models.flood_model import FloodModel, FloodRiskLevel
from src.models.scs_cn import SCSCurveNumberModel, LandUse, SoilGroup
from src.models.snowmelt import SnowmeltModel
from src.visualization.plots import (
    plot_hydrograph, plot_simulation_results, 
    plot_flood_events, create_dashboard, plot_terrain
)
from src.utils.helpers import generate_sample_data, calculate_statistics

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Все модули успешно загружены!")

## 2. Обзор регионов Кыргызстана

In [ ]:
# Вывод информации о доступных регионах
print("Доступные регионы для моделирования:")
print("=" * 60)

for code, config in REGIONS.items():
    print(f"\n{code}: {config.name}")
    print(f"  Площадь: {config.area_km2:,} км²")
    print(f"  Средняя высота: {config.avg_elevation} м")
    print(f"  Основные реки: {', '.join(config.main_rivers)}")

## 3. Генерация данных об осадках

In [ ]:
# Выбор региона для моделирования
REGION = 'chui'  # Чуйская область

# Генерация синтетических данных об осадках
loader = PrecipitationDataLoader(region=REGION)
precip_data = loader.load_synthetic(
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
    include_extreme_events=True
)

print(f"Сгенерировано {len(precip_data)} записей")
print(f"\nПервые записи:")
precip_data.head()

In [ ]:
# Получение суточных сумм
daily_totals = loader.get_daily_totals()

# Визуализация годового хода осадков
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Суточные осадки
ax = axes[0]
ax.bar(daily_totals['date'], daily_totals['mean_precip_mm'], 
       color='steelblue', alpha=0.7)
ax.set_ylabel('Осадки (мм/день)')
ax.set_title(f'Суточные осадки - {REGIONS[REGION].name}')
ax.grid(True, alpha=0.3)

# Накопленные осадки
ax = axes[1]
cumsum = daily_totals['mean_precip_mm'].cumsum()
ax.plot(daily_totals['date'], cumsum, 'b-', linewidth=2)
ax.fill_between(daily_totals['date'], 0, cumsum, alpha=0.3)
ax.set_ylabel('Накопленные осадки (мм)')
ax.set_xlabel('Дата')
ax.set_title('Накопленные осадки за год')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nГодовая сумма осадков: {cumsum.iloc[-1]:.0f} мм")
print(f"Максимальные суточные осадки: {daily_totals['max_precip_mm'].max():.1f} мм")

## 4. Модель SCS Curve Number

In [ ]:
# Демонстрация модели SCS-CN
scs_model = SCSCurveNumberModel(region=REGION)

# Расчет стока для различных осадков и CN
precipitations = np.linspace(0, 100, 100)
cn_values = [60, 70, 80, 90]

fig, ax = plt.subplots(figsize=(10, 6))

for cn in cn_values:
    runoff = scs_model.calculate_runoff(precipitations, cn)
    ax.plot(precipitations, runoff, label=f'CN = {cn}', linewidth=2)

# Линия 1:1
ax.plot(precipitations, precipitations, 'k--', alpha=0.5, label='P = Q')

ax.set_xlabel('Осадки P (мм)')
ax.set_ylabel('Сток Q (мм)')
ax.set_title('Зависимость стока от осадков (метод SCS-CN)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# Анализ чувствительности
sensitivity = scs_model.sensitivity_analysis(
    precipitation=50,  # мм
    cn_range=(40, 95)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Сток в зависимости от CN
ax = axes[0]
ax.plot(sensitivity['cn_values'], sensitivity['runoff_mm'], 'b-', linewidth=2)
ax.fill_between(sensitivity['cn_values'], 0, sensitivity['runoff_mm'], alpha=0.3)
ax.set_xlabel('Номер кривой стока (CN)')
ax.set_ylabel('Сток (мм)')
ax.set_title('Сток при P = 50 мм')
ax.grid(True, alpha=0.3)

# Коэффициент стока
ax = axes[1]
ax.plot(sensitivity['cn_values'], sensitivity['runoff_coefficient'] * 100, 'r-', linewidth=2)
ax.set_xlabel('Номер кривой стока (CN)')
ax.set_ylabel('Коэффициент стока (%)')
ax.set_title('Коэффициент стока')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Анализ рельефа

In [ ]:
# Создание синтетической модели рельефа
terrain = create_sample_terrain(region=REGION)

# Статистика рельефа
stats = terrain.get_terrain_statistics()
print("Характеристики рельефа:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.1f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Визуализация рельефа
fig = plot_terrain(
    terrain.dem,
    terrain.slope,
    terrain.flow_accumulation,
    title=f'Рельеф - {REGIONS[REGION].name}'
)
plt.show()

## 6. Моделирование паводков

In [ ]:
# Подготовка данных для моделирования
simulation_data = generate_sample_data(
    region=REGION,
    days=365,
    scenario='normal'
)

print(f"Период моделирования: {simulation_data['timestamp'].min()} - {simulation_data['timestamp'].max()}")
print(f"\nСтатистика осадков:")
print(f"  Сумма: {simulation_data['precipitation_mm'].sum():.0f} мм")
print(f"  Среднее: {simulation_data['precipitation_mm'].mean():.1f} мм/день")
print(f"  Максимум: {simulation_data['precipitation_mm'].max():.1f} мм/день")

In [ ]:
# Создание и инициализация модели
model = FloodModel(region=REGION)
model.initialize_state(
    timestamp=simulation_data['timestamp'].iloc[0],
    soil_moisture=0.4,
    snow_water_equivalent=100  # Начальный снегозапас
)

# Запуск моделирования
results = model.run_simulation(simulation_data, dt_hours=24)

print("Моделирование завершено!")
print(f"\nРасчетных шагов: {len(results)}")
print(f"Максимальный расход: {results['discharge_m3s'].max():.1f} м³/с")
print(f"Средний расход: {results['discharge_m3s'].mean():.1f} м³/с")

In [ ]:
# Выявление паводковых событий
events = model.detect_flood_events(results, min_duration_hours=6)

print(f"\nОбнаружено паводковых событий: {len(events)}")

if events:
    print("\nХарактеристики событий:")
    print("-" * 60)
    for i, event in enumerate(events[:5], 1):
        print(f"\nСобытие {i}:")
        print(f"  Начало: {event.start_time}")
        print(f"  Пик: {event.peak_time}")
        print(f"  Продолжительность: {event.duration_hours:.0f} ч")
        print(f"  Пиковый расход: {event.peak_discharge_m3s:.1f} м³/с")
        print(f"  Уровень риска: {event.risk_level.value}")

In [ ]:
# Визуализация результатов моделирования
fig = plot_simulation_results(
    results,
    variables=['precipitation_mm', 'discharge_m3s', 'surface_runoff_mm'],
    title=f'Результаты моделирования - {REGIONS[REGION].name}'
)
plt.show()

In [ ]:
# Гидрограф с осадками
threshold = model.params.flood_discharge_threshold

fig = plot_hydrograph(
    results['timestamp'],
    results['discharge_m3s'],
    results['precipitation_mm'],
    threshold=threshold,
    title=f'Гидрограф стока - {REGIONS[REGION].name}'
)
plt.show()

In [ ]:
# Визуализация паводковых событий
if events:
    fig = plot_flood_events(events, title='Анализ паводковых событий')
    plt.show()

## 7. Сценарный анализ

In [ ]:
# Сравнение сценариев
scenarios = ['dry', 'normal', 'wet', 'extreme']
scenario_results = {}

for scenario in scenarios:
    # Генерация данных
    data = generate_sample_data(region=REGION, days=365, scenario=scenario)
    
    # Моделирование
    model = FloodModel(region=REGION)
    model.initialize_state(data['timestamp'].iloc[0])
    res = model.run_simulation(data)
    evts = model.detect_flood_events(res)
    
    scenario_results[scenario] = {
        'total_precip': data['precipitation_mm'].sum(),
        'max_discharge': res['discharge_m3s'].max(),
        'mean_discharge': res['discharge_m3s'].mean(),
        'events': len(evts),
        'results': res
    }

# Вывод сравнения
print("\nСравнение сценариев:")
print("=" * 70)
print(f"{'Сценарий':<12} {'Осадки (мм)':<15} {'Макс.Q (м³/с)':<15} {'Ср.Q (м³/с)':<15} {'События':<10}")
print("-" * 70)

for scenario, data in scenario_results.items():
    print(f"{scenario:<12} {data['total_precip']:<15.0f} {data['max_discharge']:<15.1f} "
          f"{data['mean_discharge']:<15.1f} {data['events']:<10}")

In [ ]:
# Визуализация сравнения
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'dry': 'orange', 'normal': 'blue', 'wet': 'green', 'extreme': 'red'}
labels_ru = {'dry': 'Сухой', 'normal': 'Нормальный', 'wet': 'Влажный', 'extreme': 'Экстремальный'}

# Гидрографы
ax = axes[0, 0]
for scenario, data in scenario_results.items():
    res = data['results']
    ax.plot(res['timestamp'], res['discharge_m3s'], 
            color=colors[scenario], label=labels_ru[scenario], alpha=0.7)
ax.set_ylabel('Расход (м³/с)')
ax.set_title('Сравнение гидрографов')
ax.legend()
ax.grid(True, alpha=0.3)

# Столбчатая диаграмма осадков
ax = axes[0, 1]
x = range(len(scenarios))
precips = [scenario_results[s]['total_precip'] for s in scenarios]
ax.bar(x, precips, color=[colors[s] for s in scenarios])
ax.set_xticks(x)
ax.set_xticklabels([labels_ru[s] for s in scenarios])
ax.set_ylabel('Осадки (мм/год)')
ax.set_title('Годовые осадки по сценариям')

# Максимальные расходы
ax = axes[1, 0]
max_q = [scenario_results[s]['max_discharge'] for s in scenarios]
ax.bar(x, max_q, color=[colors[s] for s in scenarios])
ax.set_xticks(x)
ax.set_xticklabels([labels_ru[s] for s in scenarios])
ax.set_ylabel('Расход (м³/с)')
ax.set_title('Максимальные расходы')

# Число событий
ax = axes[1, 1]
n_events = [scenario_results[s]['events'] for s in scenarios]
ax.bar(x, n_events, color=[colors[s] for s in scenarios])
ax.set_xticks(x)
ax.set_xticklabels([labels_ru[s] for s in scenarios])
ax.set_ylabel('Число событий')
ax.set_title('Паводковые события')

plt.suptitle('Сценарный анализ паводков', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Моделирование таяния снега

In [ ]:
# Моделирование весеннего снеготаяния
snowmelt_model = SnowmeltModel(region=REGION)
snowmelt_model.initialize_snowpack(snow_water_equivalent=300)  # 300 мм снегозапаса

# Генерация температур для весеннего периода (март-май)
days = 90
day_of_year = np.arange(60, 60 + days)  # С начала марта
temperatures = 5 + 15 * (day_of_year - 60) / 90 + np.random.normal(0, 3, days)

# Моделирование
results_snow = snowmelt_model.simulate_snow_season(
    temperature_series=temperatures,
    precipitation_series=np.zeros(days),  # Без осадков
    method='degree_day'
)

# Визуализация
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

dates = pd.date_range('2024-03-01', periods=days)

ax = axes[0]
ax.plot(dates, temperatures, 'r-', linewidth=1.5)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax.fill_between(dates, 0, temperatures, where=temperatures > 0, color='red', alpha=0.2)
ax.fill_between(dates, 0, temperatures, where=temperatures < 0, color='blue', alpha=0.2)
ax.set_ylabel('Температура (°C)')
ax.set_title('Температура воздуха')

ax = axes[1]
ax.plot(dates, results_snow['snow_water_equivalent'], 'b-', linewidth=2)
ax.fill_between(dates, 0, results_snow['snow_water_equivalent'], alpha=0.3)
ax.set_ylabel('Снегозапас (мм)')
ax.set_title('Динамика снегозапаса')

ax = axes[2]
ax.bar(dates, results_snow['snowmelt'], color='cyan', alpha=0.7)
ax.set_ylabel('Таяние (мм/день)')
ax.set_xlabel('Дата')
ax.set_title('Интенсивность таяния снега')

plt.tight_layout()
plt.show()

print(f"\nНачальный снегозапас: 300 мм")
print(f"Растаяло за период: {results_snow['snowmelt'].sum():.0f} мм")
print(f"Остаток: {results_snow['snow_water_equivalent'][-1]:.0f} мм")

## 9. Информационная панель

In [ ]:
# Создание информационной панели для нормального сценария
normal_results = scenario_results['normal']['results']

# Получение событий для нормального сценария
model = FloodModel(region=REGION)
model.initialize_state(normal_results['timestamp'].iloc[0])
_ = model.run_simulation(generate_sample_data(region=REGION, days=365, scenario='normal'))
normal_events = model.detect_flood_events(_)

fig = create_dashboard(
    normal_results,
    events=normal_events,
    region=REGION
)
plt.show()

## 10. Выводы

В данном ноутбуке продемонстрированы основные возможности разработанной системы моделирования паводков:

1. **Генерация данных об осадках** - создание синтетических данных с учетом сезонных вариаций и экстремальных событий

2. **Модель SCS-CN** - расчет поверхностного стока на основе номера кривой стока

3. **Анализ рельефа** - генерация и анализ цифровой модели рельефа, расчет уклонов и направлений стока

4. **Комплексная модель паводков** - интеграция всех компонентов для моделирования паводковых процессов

5. **Сценарный анализ** - сравнение результатов для различных климатических сценариев

6. **Моделирование снеготаяния** - учет вклада талых вод в формирование паводков